In [41]:
import torch as torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 4
max_iters = 1000
learning_rate = 3e-4
eval_iters = 250

cpu


In [22]:
with open('wizard_of_oz.txt', 'r', encoding='utf=8') as f:
    text = f.read()
chars = sorted(set(text))
vocabulary_size = len(chars)
print(chars)
print(vocabulary_size)

['\n', ' ', '!', '#', '$', '%', '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']
86


In [23]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([29, 60, 53, 68, 72, 57, 70,  1, 35,  0, 46, 60, 57,  1, 29, 77, 55, 64,
        67, 66, 57,  0,  0,  0, 30, 67, 70, 67, 72, 60, 77,  1, 64, 61, 74, 57,
        56,  1, 61, 66,  1, 72, 60, 57,  1, 65, 61, 56, 71, 72,  1, 67, 58,  1,
        72, 60, 57,  1, 59, 70, 57, 53, 72,  1, 37, 53, 66, 71, 53, 71,  1, 68,
        70, 53, 61, 70, 61, 57, 71, 10,  1, 75, 61, 72, 60,  1, 47, 66, 55, 64,
        57,  0, 34, 57, 66, 70, 77, 10,  1, 75])


In [24]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [25]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('input is ', context, ' when target is ', target)

input is  tensor([29])  when target is  tensor(60)
input is  tensor([29, 60])  when target is  tensor(53)
input is  tensor([29, 60, 53])  when target is  tensor(68)
input is  tensor([29, 60, 53, 68])  when target is  tensor(72)
input is  tensor([29, 60, 53, 68, 72])  when target is  tensor(57)
input is  tensor([29, 60, 53, 68, 72, 57])  when target is  tensor(70)
input is  tensor([29, 60, 53, 68, 72, 57, 70])  when target is  tensor(1)
input is  tensor([29, 60, 53, 68, 72, 57, 70,  1])  when target is  tensor(35)


In [26]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([223953]) torch.int64
tensor([29, 60, 53, 68, 72, 57, 70,  1, 35,  0, 46, 60, 57,  1, 29, 77, 55, 64,
        67, 66, 57,  0,  0,  0, 30, 67, 70, 67, 72, 60, 77,  1, 64, 61, 74, 57,
        56,  1, 61, 66,  1, 72, 60, 57,  1, 65, 61, 56, 71, 72,  1, 67, 58,  1,
        72, 60, 57,  1, 59, 70, 57, 53, 72,  1, 37, 53, 66, 71, 53, 71,  1, 68,
        70, 53, 61, 70, 61, 57, 71, 10,  1, 75, 61, 72, 60,  1, 47, 66, 55, 64,
        57,  0, 34, 57, 66, 70, 77, 10,  1, 75])


In [27]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    dat = train_data if split == 'train' else val_data
    ix = torch.randint(len(dat) - block_size, (batch_size,))
    x = torch.stack([dat[i:i+block_size]     for i in ix])
    y = torch.stack([dat[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch('train')
print('inputs')
print(x)
print('targets')
print(y)



inputs
tensor([[61, 72, 72, 64, 57,  1, 53, 66],
        [67, 10,  1, 64, 67, 66, 59,  1],
        [60, 57,  1, 72, 70, 57, 57,  1],
        [72, 60, 57,  1, 33, 70, 57, 53]])
targets
tensor([[72, 72, 64, 57,  1, 53, 66, 61],
        [10,  1, 64, 67, 66, 59,  1, 54],
        [57,  1, 72, 70, 57, 57,  1, 65],
        [60, 57,  1, 33, 70, 57, 53, 72]])


In [43]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            # ottengo la parte "indovinata"
            logits, loss = self.forward(index)
            # mi concentro solo sull'ultimo step
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = BigramLanguageModel(vocabulary_size)
m = model.to(device)
context = torch.zeros((1, 1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
            


%USBC™-3/—‘lZH‘cY”BD#syO“RIxz9k
J?SzA”nfmE?eUnC—AcjmK%‘Zpom‘
VVuPMWNQsh’4kTkXt;0’eI8“ao*
bz;V34RaXJ2C-3h™PD2V/#2/mhoedYqMCiMC#s4XPRUudq) p%•$Sfm4HKdrGv-Ote™
pKX8LGWpz28#’x6h’2
2”
2Zp’s‘
y5-jGM
/U7’p5x”G5?“”*3Au*
o*P6’GSZ™SC*y“g7XF•!(”N ™“’—2“FN8,“iMap7!#$™nPMC4ZHH) .oQ-mu.G4C*‘A#:YqVAGua/6x%z#Ff
J™XG”nT8u”33d(P5Jvx
bEg—“PoAf*#HpdH3x!—7‘
)!‘7+7“MfJ™p%E”-6.GVCD:-GvH—9”6o,pzGv’p6Eg!N4XuJ5
zbnu‘zTo
Kxww;.q;mh?z6?%Z#•oQ+NvGjW9rqcLC;vWe5•9OR0’FXdB;GGGhdVY+mXAc#Ffg.qtf;y8chFkQ“r7afmYuUR
pwEj5d(m?Vqz™ R


In [48]:
#creo un optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    #prendo un apiccola parte di data
    xb, yb = get_batch('train')

    #valutazione loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())


4.453283309936523


In [49]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


w$#1d0m—r2FPp5-%OPjcdjwnf1-cYLg81Ef”rrhJKjg jIv’TUnX!—d/UKv9aU!‘.;ZsrEZj4k!3FKaUPsMP6sYYx86??h HH#2”C)CNdJ2”8Uz(D7x5B#FMC)+h+h1zsLIZ:QNRX5“M
6JbGdorTa)8T9Y—p52’g#voo—gaR
erTx3Sq•Uwz-’C4evh7J™Pvx7HOpnwtQ:SH)ZpPBV0Nu”(VC#dkH!Izzfmj™!kqJ)’9,•+A-%OLg!ZZm*QR70Rafl#
GvbwgF-v!R(JAQYtHHxK9eA TeYm—70JsMLdy2LeE5A2rtEjGv,
!4kVcKPEg”GHMD•FMC-QC*6jYmE7‘l TaAcfYW?poz MX“N+Z%yy/ ,p+7g1.;Zt
f$fm0M5-“X•t:S™imb04H”!rKv:F•FiVY,ZkD3’B;GTBS6H”‘hy1””
.bZtn-
JliMDIfim?rVxL %OjlAQNI4sOvh/HN62fCm—Z3A,XHrTCAwl•-
wWDFr?ui
